# RAG chatbot demonstration
This notebook demonstates the capability of a chat bot with RAG

In [21]:
import os
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [12]:
# 1. Read el_state table where process='gold' and status 'R'
el_state_df = get_sql_df("SELECT * FROM state.el_state WHERE process = 'gold' AND status = 'R' ORDER BY process_ts DESC LIMIT 1")
display(el_state_df)
latest_process_ts = el_state_df['process_ts'].iloc[0] if not el_state_df.empty else None
print(f"Latest gold process_ts: {latest_process_ts}")

,process,process_ts,start_timestamp,end_timestamp,status,record_count,error_message
0,gold,1777136466868,2026-04-25 22:31:13.454732+05:30,2026-04-25 22:31:19.401462+05:30,R,8,None


Latest gold process_ts: 1777136466868


In [13]:
# 2. Read data from gold tables where process_ts from above el_state table
if latest_process_ts:
    gold_mix_df = print_sql_df(f"SELECT * FROM gold.daily_electricity_mix WHERE process_ts = {latest_process_ts} LIMIT 10")
    gold_imports_df = print_sql_df(f"SELECT * FROM gold.daily_imports WHERE process_ts = {latest_process_ts} LIMIT 10")
    gold_exports_df = print_sql_df(f"SELECT * FROM gold.daily_exports WHERE process_ts = {latest_process_ts} LIMIT 10")
else:
    print("No latest process_ts found")

,process_ts,zone,zone_name,date,nuclear_pct,biomass_pct,wind_pct,solar_pct,hydro_pct,gas_pct,...,coal_pct,geothermal_pct,unknown_pct,total_production_mwh,fossil_free_avg_pct,renewable_avg_pct,hours_covered,year,month,day
0,1777136466868,FR,France,2026-04-25,73.935531,2.202645,3.139444,11.534831,8.443589,0.672814,...,0.0,0.0,0.0,847009.645,99.25604,25.320509,16,2026,4,25


,process_ts,zone,zone_name,source_zone,source_zone_name,date,import_mwh,hours_covered,year,month,day
0,1777136466868,FR,France,BE,Belgium,2026-04-25,39728.0,16,2026,4,25
1,1777136466868,FR,France,ES,Spain,2026-04-25,42045.0,16,2026,4,25
2,1777136466868,FR,France,DE,Germany,2026-04-25,12461.0,16,2026,4,25
3,1777136466868,FR,France,CH,Switzerland,2026-04-25,2930.0,16,2026,4,25


,process_ts,zone,zone_name,destination_zone,destination_zone_name,date,export_mwh,hours_covered,year,month,day
0,1777136466868,FR,France,LU,LU,2026-04-25,2945.0,16,2026,4,25
1,1777136466868,FR,France,IT-NO,Italy North,2026-04-25,17112.0,16,2026,4,25
2,1777136466868,FR,France,GB,Great Britain,2026-04-25,46908.0,16,2026,4,25


In [ ]:
# 3. Data schema and table details for LLM
gold_schemas = {
    "daily_electricity_mix": {
        "description": "Daily electricity mix data with percentages of different energy sources",
        "columns": ["process_ts", "zone", "zone_name", "date", "nuclear_pct", "biomass_pct", "wind_pct", "solar_pct", "hydro_pct", "gas_pct", "oil_pct", "coal_pct", "geothermal_pct", "unknown_pct", "total_production_mwh", "fossil_free_avg_pct", "renewable_avg_pct", "hours_covered", "year", "month", "day"]
    },
    "daily_imports": {
        "description": "Daily electricity imports data",
        "columns": ["process_ts", "zone", "zone_name", "source_zone", "source_zone_name", "date", "import_mwh", "hours_covered", "year", "month", "day"]
    },
    "daily_exports": {
        "description": "Daily electricity exports data",
        "columns": ["process_ts", "zone", "zone_name", "destination_zone", "destination_zone_name", "date", "export_mwh", "hours_covered", "year", "month", "day"]
    }
}

# Initialize LLM
#print(f"API Key: {settings.llm_api_key}")
print(f"Model: {settings.llm_model}")
llm = ChatGoogleGenerativeAI(api_key=settings.llm_api_key, model=settings.llm_model)

schema_context = "\n".join([f"Table: {table}\nDescription: {info['description']}\nColumns: {', '.join(info['columns'])}\n" for table, info in gold_schemas.items()])

API Key: AIzaSyCgx6........
Model: gemini-2.5-flash


In [22]:
# 4. Process docs folder and create embeddings in ChromaDB
docs_path = "../docs/Electricity_Maps_doc.pdf"
loader = PyPDFLoader(docs_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory="./chroma_db")
retriever = vectorstore.as_retriever()

C:\Users\user\AppData\Local\Temp\ipykernel_17692\1141313789.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
e:\IntelliJ\intelliWorkspace\ElectricityMaps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\IntelliJ\intelliWorkspace\ElectricityMaps\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated fil

In [24]:
# 5a. Query sample: Read from gold layer about data and display
print("Sample query: Show electricity mix for France")
sample_query = "SELECT zone, zone_name, date, renewable_avg_pct, fossil_free_avg_pct FROM gold.daily_electricity_mix WHERE zone = 'FR' ORDER BY date DESC LIMIT 5"
print_sql_df(sample_query)

Sample query: Show electricity mix for France


,zone,zone_name,date,renewable_avg_pct,fossil_free_avg_pct
0,FR,France,2026-04-25,25.320509,99.256040
1,FR,France,2026-04-25,24.984061,99.266444
2,FR,France,2026-04-24,25.441676,99.295859


,zone,zone_name,date,renewable_avg_pct,fossil_free_avg_pct
0,FR,France,2026-04-25,25.320509,99.256040
1,FR,France,2026-04-25,24.984061,99.266444
2,FR,France,2026-04-24,25.441676,99.295859


In [25]:
# 5b. Query sample: FAQ related data from vector store
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)

faq_question = "What is Electricity Maps?"
response = rag_chain.invoke(faq_question)
print(f"FAQ Question: {faq_question}")
print(f"Answer: {response.content}")

FAQ Question: What is Electricity Maps?
Answer: Electricity Maps provides real-time data for decision-making, employing a robust, triple-layered methodological choice to offer accurate and verifiable data that closely represents the physical reality of electricity grids. Its data is attributional, location-based, and consumption-based. It provides grid signals (such as electricity mix and carbon intensity) for the electricity available or consumed in a grid, utilizing more granular data for electricity exchanges between zones and countries, upstream emissions factors, and spatial and time granularity compared to other sources like the IEA. Its aim is to offer actionable data through different spatial and temporal aggregations.
